# CVA Calculation with ORE: Monte Carlo Exposure, Netting, Collateral & CDS Integration

A complete end-to-end CVA workflow built on ORE's Python bindings:

| Step | What we compute | Key component |
|:---|:---|:---|
| **1** | Monte Carlo EPE/ENE profile | `OREApp` + 1-factor LGM simulation |
| **2** | Impact of CSA threshold collateral | `OREApp` XVA re-run with active CSA |
| **3** | Counterparty default probabilities | `SpreadCdsHelper` + `PiecewiseFlatHazardRate` |
| **4** | CVA = LGD x sum EPE(t) x dPD(t) | Numerical integration in Python |

**Portfolio:** 3 interest rate swaps — GBP 40MM (2027), EUR 30MM (2026), USD 50MM (2024)  
**Netting set:** `CPTY_A` (one agreement covering all 3 swaps)  
**Counterparty ACME Corp:** recovery = 40%, CDS spreads 50-130 bps  
**As-of date:** 2016-02-05

In [1]:
import os, sys, time, math
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from ORE import *

EXAMPLE_DIR = '/home/Aloysuis/libs/ore/Examples/ExposureWithCollateral'
OUTPUT_DIR  = '/home/Aloysuis/libs/ore/Examples/CreditRisk'

def get_exposure(app, report_name):
    rep = app.getReport(report_name)
    return {
        'Time': list(rep.dataAsReal(2)),
        'EPE':  list(rep.dataAsReal(3)),
        'ENE':  list(rep.dataAsReal(4)),
    }

print('ORE loaded successfully')

ORE loaded successfully


---
## Part 1: Monte Carlo Exposure Simulation

ORE runs a cross-sectional Monte Carlo simulation using a **1-factor LGM (Linear Gaussian Markov)** model.
For each scenario the portfolio is re-priced, producing an NPV cube from which the aggregation engine computes:

- **EPE(t)** = E[max(V(t), 0)] -- expected positive exposure, drives CVA
- **ENE(t)** = E[min(V(t), 0)] -- expected negative exposure, drives DVA

The `ore.xml` config runs both simulation and XVA analytics. Collateral CSA is **disabled** here.

In [2]:
os.makedirs(os.path.join(EXAMPLE_DIR, 'Output', 'nocollateral'), exist_ok=True)
os.chdir(EXAMPLE_DIR)

params = Parameters()
params.fromFile('Input/ore.xml')

print('Running Monte Carlo simulation (uncollateralised) ...')
t0 = time.time()
ore_nocol = OREApp(params)
ore_nocol.run()
elapsed = time.time() - t0

errors = ore_nocol.getErrors()
print(f'  Completed in {elapsed:.1f}s  |  Errors: {len(errors)}')
for e in errors:
    print('  ERROR:', e)
print()
print('Available reports:')
for r in sorted(ore_nocol.getReportNames()):
    print(' -', r)

Running Monte Carlo simulation (uncollateralised) ...
  Completed in 54.5s  |  Errors: 0

Available reports:
 - cashflow
 - colva_nettingset_CPTY_A
 - curves
 - cva_sensitivity_nettingset_CPTY_A
 - dividends
 - exposure_nettingset_CPTY_A
 - exposure_trade_Swap_1
 - exposure_trade_Swap_2
 - exposure_trade_Swap_3
 - fixings
 - marketdata
 - netcube
 - npv
 - pricingstats
 - rawcube
 - runtimes
 - todaysmarketcalibration
 - todaysmarketcalibration_cashflows
 - xva


In [3]:
exp_ns = get_exposure(ore_nocol, 'exposure_nettingset_CPTY_A')
exp_s1 = get_exposure(ore_nocol, 'exposure_trade_Swap_1')
exp_s2 = get_exposure(ore_nocol, 'exposure_trade_Swap_2')
exp_s3 = get_exposure(ore_nocol, 'exposure_trade_Swap_3')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Part 1 - Monte Carlo Exposure (No Collateral)', fontsize=13, fontweight='bold')

ax = axes[0]
ax.plot(exp_s1['Time'], exp_s1['EPE'], label='Swap 1 - GBP 40MM (2027)', color='steelblue', lw=1.5)
ax.plot(exp_s2['Time'], exp_s2['EPE'], label='Swap 2 - EUR 30MM (2026)', color='tomato',    lw=1.5)
ax.plot(exp_s3['Time'], exp_s3['EPE'], label='Swap 3 - USD 50MM (2024)', color='seagreen',  lw=1.5)
ax.plot(exp_ns['Time'], exp_ns['EPE'], label='Netting Set CPTY_A',        color='darkorange', lw=2.5, ls='--')
ax.set_xlabel('Time (years)'); ax.set_ylabel('EPE (EUR)')
ax.set_title('Individual Trade EPE vs Netting Set EPE')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(exp_ns['Time'], exp_ns['EPE'], label='EPE', color='royalblue', lw=2)
ax.plot(exp_ns['Time'], exp_ns['ENE'], label='ENE', color='crimson',   lw=2)
ax.fill_between(exp_ns['Time'], exp_ns['EPE'], 0, alpha=0.15, color='royalblue')
ax.fill_between(exp_ns['Time'], exp_ns['ENE'], 0, alpha=0.15, color='crimson')
ax.axhline(0, color='black', lw=0.8)
ax.set_xlabel('Time (years)'); ax.set_ylabel('Exposure (EUR)')
ax.set_title('Netting Set: EPE and ENE (Uncollateralised)')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'cva_exposure_nocol.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'Peak EPE (netting set): EUR {max(exp_ns["EPE"]):>12,.0f}')
print(f'Simulation horizon:     {max(exp_ns["Time"]):>8.1f} years')
print(f'Exposure grid points:   {len(exp_ns["Time"]):>8d}')

Peak EPE (netting set): EUR    3,572,203
Simulation horizon:         11.1 years
Exposure grid points:        291


---
## Part 2: Effect of CSA (Threshold Collateral)

We re-run the XVA aggregation with an **active Credit Support Annex (CSA)**:

| CSA Parameter | Value |
|:---|---:|
| Threshold (both directions) | EUR 1,000,000 |
| Minimum Transfer Amount (MTA) | EUR 100,000 |
| Margin Period of Risk (MPoR) | 2 weeks |
| Collateral currency | EUR |

The `ore_threshold.xml` config **reuses the existing NPV cube** from Part 1 and applies
CSA collateral netting logic only -- making this second run very fast.

In [4]:
os.makedirs(os.path.join(EXAMPLE_DIR, 'Output', 'vm_threshold'), exist_ok=True)

params2 = Parameters()
params2.fromFile('Input/ore_threshold.xml')

print('Running XVA with CSA threshold (reuses existing cube) ...')
t0 = time.time()
ore_thresh = OREApp(params2)
ore_thresh.run()
elapsed = time.time() - t0

errors = ore_thresh.getErrors()
print(f'  Completed in {elapsed:.1f}s  |  Errors: {len(errors)}')
for e in errors:
    print('  ERROR:', e)

Running XVA with CSA threshold (reuses existing cube) ...
  Completed in 16.8s  |  Errors: 0


In [5]:
exp_thresh = get_exposure(ore_thresh, 'exposure_nettingset_CPTY_A')

t_arr   = np.array(exp_ns['Time'])
epe_arr = np.array(exp_ns['EPE'])
thr_arr = np.array(exp_thresh['EPE'])

# Effective EPE: time-averaged EPE over first year (Basel CCR metric)
mask_1y     = t_arr <= 1.0
t_1y        = t_arr[mask_1y]
eepe_nocol  = float(np.trapezoid(epe_arr[mask_1y], t_1y)) if len(t_1y) > 1 else 0.0
eepe_thresh = float(np.trapezoid(thr_arr[mask_1y], t_1y)) if len(t_1y) > 1 else 0.0

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Part 2 - Collateral Impact (CSA Threshold EUR 1MM)', fontsize=13, fontweight='bold')

ax = axes[0]
ax.plot(t_arr, epe_arr, label='EPE - No Collateral',     color='royalblue',  lw=2)
ax.plot(t_arr, thr_arr, label='EPE - CSA Threshold 1MM', color='darkorange', lw=2, ls='--')
ax.fill_between(t_arr, epe_arr, thr_arr,
                where=(epe_arr > thr_arr), alpha=0.20, color='steelblue', label='Exposure Reduction')
ax.set_xlabel('Time (years)'); ax.set_ylabel('EPE (EUR)')
ax.set_title('EPE: No Collateral vs CSA Threshold')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

ax = axes[1]
labels  = ['No Collateral\n(EEPE 1Y)', 'CSA Threshold 1MM\n(EEPE 1Y)']
values  = [eepe_nocol, eepe_thresh]
bars = ax.bar(labels, values, color=['royalblue', 'darkorange'], alpha=0.75, width=0.45)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(values) * 0.01,
            f'EUR {val:,.0f} yr', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylabel('Effective EPE (EUR x years)'); ax.set_title('1Y EEPE - Basel CCR Metric')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'cva_collateral_effect.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'  EEPE (no collateral, 1Y):  EUR {eepe_nocol:>12,.0f} x years')
print(f'  EEPE (CSA threshold,  1Y): EUR {eepe_thresh:>12,.0f} x years')
if eepe_nocol > 0:
    print(f'  EEPE reduction:              {100 * (1 - eepe_thresh / eepe_nocol):>8.1f}%')

  EEPE (no collateral, 1Y):  EUR    2,387,515 x years
  EEPE (CSA threshold,  1Y): EUR      764,635 x years
  EEPE reduction:                  68.0%


---
## Part 3: CDS Hazard Curve Bootstrap

We build ACME Corp's default probability curve by bootstrapping **7 CDS par spread quotes**
using `PiecewiseFlatHazardRate` with `SpreadCdsHelper` (ISDA CDS 2015 convention, quarterly coupons).

The bootstrapper finds piecewise-constant hazard rates lambda_i such that each instrument reprices exactly:

    SP(t) = exp( -integral_0^t lambda(s) ds )

| Tenor | Par Spread (bps) | | Tenor | Par Spread (bps) |
|:---:|---:|-|:---:|---:|
| 6M  |  50 | | 5Y  | 120 |
| 1Y  |  75 | | 7Y  | 125 |
| 2Y  | 100 | | 10Y | 130 |
| 3Y  | 110 | | | |

In [6]:
asof     = Date(5, February, 2016)    # match ORE simulation as-of date
Settings.instance().evaluationDate = asof

recovery = 0.40
ois_rate = 0.0025    # flat EUR OIS ~25 bps (2016 environment)

cds_market = [
    (Period(6,  Months),  50),
    (Period(1,  Years),   75),
    (Period(2,  Years),  100),
    (Period(3,  Years),  110),
    (Period(5,  Years),  120),
    (Period(7,  Years),  125),
    (Period(10, Years),  130),
]

flat_ois    = FlatForward(asof, QuoteHandle(SimpleQuote(ois_rate)), Actual365Fixed())
disc_handle = YieldTermStructureHandle(flat_ois)

helpers = []
for tenor, spread_bps in cds_market:
    q = QuoteHandle(SimpleQuote(spread_bps / 10000.0))
    h = SpreadCdsHelper(
        q, tenor,
        1,
        WeekendsOnly(),
        Quarterly,
        Following,
        DateGeneration.CDS2015,
        Actual360(),
        recovery,
        disc_handle,
    )
    helpers.append(h)

hazard_curve = PiecewiseFlatHazardRate(asof, helpers, Actual365Fixed())
hazard_curve.enableExtrapolation()

a365  = Actual365Fixed()
nodes = hazard_curve.nodes()

print(f"{'Pillar Date':<16}  {'T (yrs)':>8}  {'Hazard Rate (bps)':>18}  {'Survival Prob':>14}")
print('-' * 65)
for d, h_rate in nodes:
    t  = a365.yearFraction(asof, d)
    sp = hazard_curve.survivalProbability(d)
    print(f'  {d.ISO():<14}  {t:>8.3f}  {h_rate * 10000:>18.4f}  {sp:>14.6f}')

Pillar Date        T (yrs)   Hazard Rate (bps)   Survival Prob
-----------------------------------------------------------------
  2016-02-05         0.000             85.0852        1.000000
  2016-06-20         0.373             85.0852        0.996835
  2016-12-20         0.874            158.2825        0.988955
  2017-12-20         1.874            206.5328        0.968740
  2018-12-20         2.874            218.5137        0.947801
  2020-12-21         4.879            228.4067        0.905365
  2022-12-20         6.877            233.5994        0.864094
  2025-12-22         9.885            241.5639        0.803530


In [7]:
def t_to_date(t):
    return asof + round(t * 365.25)

t_plot  = np.linspace(0.02, 11.0, 220)
sp_plot = [hazard_curve.survivalProbability(t_to_date(t)) for t in t_plot]

# Node values (skip t=0)
node_t  = [a365.yearFraction(asof, d) for d, _ in nodes[1:]]
node_h  = [h * 10000                  for _, h in nodes[1:]]
node_sp = [hazard_curve.survivalProbability(d) for d, _ in nodes[1:]]

# Step-function for hazard rate
pillar_t = [a365.yearFraction(asof, d) for d, _ in nodes]
pillar_h = [h * 10000                  for _, h in nodes]
h_step   = np.interp(t_plot, pillar_t, pillar_h, left=pillar_h[0], right=pillar_h[-1])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Part 3 - ACME Corp: CDS Hazard Curve Bootstrap', fontsize=13, fontweight='bold')

ax1.step(t_plot, h_step, color='steelblue', lw=2, where='post', label='Piecewise-flat hazard rate')
ax1.scatter(node_t, node_h, color='darkorange', zorder=5, s=60, label='Bootstrap pillar')
ax1.set_xlabel('Time (years)'); ax1.set_ylabel('Hazard Rate (bps)')
ax1.set_title('Piecewise-Flat Hazard Rate lambda(t)')
ax1.legend(fontsize=9); ax1.grid(alpha=0.3)

ax2.plot(t_plot, sp_plot, color='steelblue', lw=2, label='Survival probability SP(t)')
ax2.scatter(node_t, node_sp, color='darkorange', zorder=5, s=60, label='Bootstrap pillar')
ax2.axhline(recovery, ls=':', color='crimson', lw=1.5, label=f'Recovery R = {recovery:.0%}')
ax2.set_xlabel('Time (years)'); ax2.set_ylabel('Probability')
ax2.set_title('Survival Probability SP(t)')
ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'cva_cds_bootstrap.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## Part 4: CVA Calculation

Standard discrete CVA formula:

    CVA = (1-R) * sum_i  DF(t_i) * EPE(t_i) * [SP(t_{i-1}) - SP(t_i)]

where delta_PD(t_i) = SP(t_{i-1}) - SP(t_i) is the marginal default probability
in bucket [t_{i-1}, t_i], and LGD = 1 - R = 60%.

We compare CVA under:
- **Uncollateralised** exposure profile (Part 1)
- **CSA threshold EUR 1MM** exposure profile (Part 2)

In [8]:
def compute_cva(time_arr, epe_arr, haz_crv, disc_crv, asof_date, rec=0.40):
    lgd     = 1.0 - rec
    cva     = 0.0
    sp_prev = 1.0
    rows    = []
    for t, epe in zip(time_arr, epe_arr):
        if t <= 0:
            continue
        d       = asof_date + round(t * 365.25)
        sp      = haz_crv.survivalProbability(d)
        df      = disc_crv.discount(d)
        dpd     = max(sp_prev - sp, 0.0)
        contrib = lgd * df * epe * dpd
        cva    += contrib
        rows.append({'time': t, 'epe': epe, 'sp': sp, 'df': df,
                     'marginal_pd': dpd, 'contribution': contrib})
        sp_prev = sp
    return cva, pd.DataFrame(rows)

cva_nocol,  df_cva_nocol  = compute_cva(
    exp_ns['Time'],     exp_ns['EPE'],     hazard_curve, disc_handle, asof)
cva_thresh, df_cva_thresh = compute_cva(
    exp_thresh['Time'], exp_thresh['EPE'], hazard_curve, disc_handle, asof)

print('=' * 52)
print(f'  CVA (uncollateralised):    EUR {cva_nocol:>12,.2f}')
print(f'  CVA (CSA threshold 1MM):   EUR {cva_thresh:>12,.2f}')
print('-' * 52)
if cva_nocol > 0:
    reduction = 100 * (1 - cva_thresh / cva_nocol)
    print(f'  CVA reduction from CSA:       {reduction:>7.1f}%')
print('=' * 52)

  CVA (uncollateralised):    EUR   302,205.98
  CVA (CSA threshold 1MM):   EUR    65,229.43
----------------------------------------------------
  CVA reduction from CSA:          78.4%


In [9]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Part 4 - CVA Calculation Summary', fontsize=13, fontweight='bold')

# Panel A: EPE comparison
ax = axes[0]
ax.plot(exp_ns['Time'],     exp_ns['EPE'],     color='royalblue',  lw=2, label='Uncollateralised')
ax.plot(exp_thresh['Time'], exp_thresh['EPE'], color='darkorange', lw=2, ls='--', label='CSA Threshold 1MM')
epe_np  = np.array(exp_ns['EPE'])
thr_np  = np.array(exp_thresh['EPE'])
t_np    = np.array(exp_ns['Time'])
ax.fill_between(t_np, epe_np, thr_np, where=(epe_np > thr_np), alpha=0.15, color='steelblue')
ax.set_xlabel('Time (years)'); ax.set_ylabel('EPE (EUR)')
ax.set_title('Exposure Profile EPE(t)')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# Panel B: cumulative CVA
ax = axes[1]
cum_nocol  = df_cva_nocol['contribution'].values.cumsum()
cum_thresh = df_cva_thresh['contribution'].values.cumsum()
ax.plot(df_cva_nocol['time'],  cum_nocol,  color='royalblue',  lw=2,
        label=f'Uncollateralised  EUR {cva_nocol:,.0f}')
ax.plot(df_cva_thresh['time'], cum_thresh, color='darkorange', lw=2, ls='--',
        label=f'CSA Threshold 1MM  EUR {cva_thresh:,.0f}')
ax.fill_between(df_cva_nocol['time'], cum_nocol, cum_thresh,
                where=(cum_nocol > cum_thresh), alpha=0.15, color='royalblue')
ax.set_xlabel('Time (years)'); ax.set_ylabel('Cumulative CVA (EUR)')
ax.set_title('Cumulative CVA Build-Up')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# Panel C: bar chart
ax = axes[2]
values = [cva_nocol, cva_thresh]
bars = ax.bar(['Uncollateralised', 'CSA Threshold EUR 1MM'],
              values, color=['royalblue', 'darkorange'], alpha=0.75, width=0.45)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + max(values) * 0.01,
            f'EUR {val:,.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
if cva_nocol > 0:
    reduct = 100 * (1 - cva_thresh / cva_nocol)
    ax.text(0.5, (cva_nocol + cva_thresh) / 2,
            f'down {reduct:.1f}%\nCSA reduction',
            ha='center', va='center', fontsize=10, color='darkgreen', fontweight='bold',
            transform=ax.get_yaxis_transform())
ax.set_ylabel('CVA (EUR)'); ax.set_title('CVA: Collateral Impact')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'cva_summary.png'), dpi=150, bbox_inches='tight')
plt.show()

# Summary table
summary = pd.DataFrame({
    'Scenario':         ['No Collateral', 'CSA Threshold EUR 1MM'],
    'Peak EPE (EUR)':   [f"{max(exp_ns['EPE']):,.0f}",     f"{max(exp_thresh['EPE']):,.0f}"],
    'EEPE 1Y (EUR yr)': [f"{eepe_nocol:,.0f}",             f"{eepe_thresh:,.0f}"],
    'CVA (EUR)':        [f"{cva_nocol:,.0f}",              f"{cva_thresh:,.0f}"],
})
display(summary.style.set_caption('CVA Summary').hide(axis='index'))

Scenario,Peak EPE (EUR),EEPE 1Y (EUR yr),CVA (EUR)
No Collateral,"3,572,203","2,387,515","302,206"
CSA Threshold EUR 1MM,"1,201,478","764,635","65,229"
